In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

access_id = os.getenv("ODPS_ACCESS_ID")
access_key = os.getenv("ODPS_ACCESS_KEY")
project = os.getenv("ODPS_PROJECT")
endpoint = os.getenv("ODPS_ENDPOINT")

from odps import ODPS, options

o = ODPS(
    access_id=access_id,
    secret_access_key=access_key,
    project=project,
    endpoint=endpoint,
)

options.tunnel.use_instance_tunnel = True
options.tunnel.limit_instance_tunnel = False


In [ ]:
def q(sql):
    """Run SQL on DataWorks and return the result as a pandas DataFrame."""
    inst = o.execute_sql(sql)
    inst.wait_for_success()
    with inst.open_reader(tunnel=True) as reader:
        return reader.to_pandas()


print("Connected to project:", o.project)
print(os.getenv("ODPS_ACCESS_ID"))
print(os.getenv("ODPS_ACCESS_KEY"))

In [ ]:
sql = f"""
SELECT * FROM BaseHub_mx_dev.stolencard_dataset_all_tables;
"""

instance = o.execute_sql(sql)
instance.wait_for_success()

In [ ]:
import pandas as pd

with instance.open_reader(tunnel=True) as reader:
    df = reader.to_pandas()

df

### Missing and Coverage Rate

In [ ]:
import numpy as np
import pandas as pd

def variable_report(data: pd.DataFrame, placeholders: list | None = None) -> pd.DataFrame:
    """
    Profile every variable in df.

    Parameters
    ----------
    placeholders : values that count as "no real data" for coverage
                   (e.g. ["", "N/A", -999]). Nulls are always excluded.
    """
    
    rows = []
    for col in data.columns:
        s = data[col]
        missing_rate = s.isna().mean()
        valid = s.notna() & ~s.isin(placeholders)
        coverage_rate = valid.mean()

        rows.append({
            "variable": col,
            "missing_rate": round(missing_rate, 4),
            "coverage_rate": round(coverage_rate, 4)
        })

    return pd.DataFrame(rows)


quality_table = variable_report(df, placeholders=["", "N/A"])
quality_table.to_csv('quality_table.csv')
print(quality_table)

In [ ]:
quality_table.to_csv('quality_table.csv')

### Stability

In [ ]:
import pandas as pd

MONTH_NAMES = {7: "July", 8: "August", 9: "September"}


def monthly_stats(
    df: pd.DataFrame,
    date_col: str,
    trans_id_col: str = "trans_id",
    year: int | None = None,
    exclude_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Summarize every variable per month (July, August and September).

    Returns a long-format DataFrame with one row per (variable, month) and columns:
    total_count, count_variable, nulls, perc_nulls, mean, min, p25, median, p75, max.

    - total_count = count of `trans_id_col` in the month (same for every variable).
    - count_variable = non-null values of the variable in the month.
    - nulls = (total_count - count_variable) / total_count.
    - perc_nulls = nulls / total_count.
    - Non-numeric columns only get the count/null columns (other stats are NaN).
    - `trans_id_col` and `date_col` are not summarized as variables.
    - `year`: if None, the latest year found in `date_col` is used.
    """
    exclude = set(exclude_cols or []) | {date_col, trans_id_col}

    dates = pd.to_datetime(df[date_col], errors="coerce")
    if year is None:
        year = int(dates.dt.year.max())

    mask = (dates.dt.year == year) & dates.dt.month.isin([7, 8, 9])
    data = df.loc[mask]
    month = dates[mask].dt.month.rename("month")

    # total transactions per month (from trans_id)
    total_count = data[trans_id_col].groupby(month, observed=True).count()

    sub = data.drop(columns=[c for c in exclude if c in data.columns])
    cols = list(sub.columns)
    num_cols = list(sub.select_dtypes(include="number").columns)

    g = sub.groupby(month, observed=True)

    # count of values per variable, and nulls (vectorized for ALL columns)
    count_variable = g[cols].count()
    nulls = count_variable.rsub(total_count, axis=0)  # (total - count_variable) / total
    perc_nulls = nulls.div(total_count, axis=0)                                # nulls / total

    stats = {"count_variable": count_variable, "nulls": nulls, "perc_nulls": perc_nulls}

    # numeric-only stats
    if num_cols:
        gn = g[num_cols]
        agg = gn.agg(["mean", "min", "max"])
        stats["mean"] = agg.xs("mean", axis=1, level=1)
        stats["min"] = agg.xs("min", axis=1, level=1)
        stats["max"] = agg.xs("max", axis=1, level=1)

        q = gn.quantile([0.25, 0.5, 0.75])  # index: (month, quantile)
        stats["p25"] = q.xs(0.25, level=1)
        stats["median"] = q.xs(0.5, level=1)
        stats["p75"] = q.xs(0.75, level=1)

    # long format: one row per (variable, month)
    long = pd.concat(stats, names=["stat"]).stack(future_stack=True)
    long.index.names = ["stat", "month", "variable"]
    out = long.unstack("stat").reset_index()

    out["total_count"] = out["month"].map(total_count)

    final_cols = ["total_count", "count_variable", "nulls", "perc_nulls",
                  "mean", "min", "p25", "median", "p75", "max"]
    for c in final_cols:
        if c not in out:
            out[c] = float("nan")

    out["month"] = out["month"].map(MONTH_NAMES)
    out["month"] = pd.Categorical(out["month"], ["July", "August", "September"], ordered=True)
    out = out[["variable", "month"] + final_cols]
    return out.sort_values(["variable", "month"]).reset_index(drop=True)

In [ ]:
stability = monthly_stats(df, date_col = 'transaction_time', trans_id_col = "trans_id", year = 2026)
stability.to_csv('stability.csv')
stability

In [ ]:
import numpy as np
import pandas as pd

MONTH_NAMES = {7: "July", 8: "August", 9: "September"}
STAT_COLS = ["total_count", "count_variable", "nulls", "perc_nulls",
             "mean", "min", "p25", "median", "p75", "max", "iv"]


def calculate_woe_iv(df, feature, target):
    # 1. Create a contingency table of counts
    grouped = df.groupby(feature)[target].agg(
        goods=lambda x: (x == 1).sum(), bads=lambda x: (x == 0).sum()
    )

    # 2. Calculate percentages of total goods and bads
    total_goods = grouped["goods"].sum()
    total_bads = grouped["bads"].sum()

    grouped["pct_goods"] = grouped["goods"] / total_goods
    grouped["pct_bads"] = grouped["bads"] / total_bads

    # 3. Handle zero divisions by adding a small adjustment if necessary
    grouped["pct_goods"] = grouped["pct_goods"].replace(0, 0.0001)
    grouped["pct_bads"] = grouped["pct_bads"].replace(0, 0.0001)

    # 4. Calculate WoE and IV
    grouped["WoE"] = np.log(grouped["pct_goods"] / grouped["pct_bads"])
    grouped["IV_bin"] = (grouped["pct_goods"] - grouped["pct_bads"]) * grouped[
        "WoE"
    ]

    total_iv = grouped["IV_bin"].sum()

    return grouped, total_iv


def _bin_feature(x: pd.Series, bins: int, max_cats: int):
    """
    Turns a column into groups so calculate_woe_iv can work on it:
    - numeric with more than `bins` distinct values -> quantile bins
    - other numeric / text -> each value is its own group
    - nulls -> their own group ("missing")
    Returns None when the column can't be grouped sensibly
    (datetime, or text with more than `max_cats` categories).
    """
    if pd.api.types.is_datetime64_any_dtype(x) or pd.api.types.is_timedelta64_dtype(x):
        return None
    is_numeric = pd.api.types.is_numeric_dtype(x) and not pd.api.types.is_bool_dtype(x)
    if is_numeric:
        if x.nunique() > bins:
            x = pd.qcut(x, bins, labels=False, duplicates="drop")
    elif x.nunique() > max_cats:
        return None
    return x.astype(object).fillna("missing").astype(str)


def monthly_stats(
    df: pd.DataFrame,
    date_col: str,
    trans_id_col: str = "trans_id",
    year: int | None = None,
    exclude_cols: list[str] | None = None,
    target_col: str = "fraud_label",
    bins: int = 10,
    max_cats: int = 20,
) -> pd.DataFrame:
    """
    Summarize every variable per month (July, August and September).

    Returns a long-format DataFrame with one row per (variable, month) and columns:
    total_count, count_variable, nulls, perc_nulls, mean, min, p25, median, p75, max, iv.

    - total_count = count of `trans_id_col` in the month (same for every variable).
    - count_variable = non-null values of the variable in the month.
    - nulls = (total_count - count_variable) / total_count.
    - perc_nulls = nulls / total_count.
    - Non-numeric columns only get the count/null columns and iv (other stats are NaN).
    - iv = Information Value vs `target_col` (0/1), computed inside each month with
      calculate_woe_iv. Numeric variables are cut into `bins` quantile bins first.
    - `date_col`, `trans_id_col` and `target_col` are not summarized as variables.
    - `year`: if None, the latest year found in `date_col` is used.
    """
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe")

    exclude = set(exclude_cols or []) | {date_col, trans_id_col, target_col}

    dates = pd.to_datetime(df[date_col], errors="coerce")
    if year is None:
        year = int(dates.dt.year.max())

    mask = (dates.dt.year == year) & dates.dt.month.isin([7, 8, 9])
    data = df.loc[mask]
    y = pd.to_numeric(data[target_col], errors="coerce")
    month = dates[mask].dt.month.rename("month")

    # total transactions per month (from trans_id)
    total_count = data[trans_id_col].groupby(month, observed=True).count()

    sub = data.drop(columns=[c for c in exclude if c in data.columns])
    cols = list(sub.columns)
    num_cols = list(sub.select_dtypes(include="number").columns)

    g = sub.groupby(month, observed=True)

    # count of values per variable, and nulls (vectorized for ALL columns)
    count_variable = g[cols].count()
    nulls = count_variable.rsub(total_count, axis=0)  # (total - count_variable) / total
    perc_nulls = nulls.div(total_count, axis=0)                                # nulls / total

    stats = {"count_variable": count_variable, "nulls": nulls, "perc_nulls": perc_nulls}

    # numeric-only stats
    if num_cols:
        gn = g[num_cols]
        agg = gn.agg(["mean", "min", "max"])
        stats["mean"] = agg.xs("mean", axis=1, level=1)
        stats["min"] = agg.xs("min", axis=1, level=1)
        stats["max"] = agg.xs("max", axis=1, level=1)

        q = gn.quantile([0.25, 0.5, 0.75])  # index: (month, quantile)
        stats["p25"] = q.xs(0.25, level=1)
        stats["median"] = q.xs(0.5, level=1)
        stats["p75"] = q.xs(0.75, level=1)

    # long format: one row per (variable, month)
    long = pd.concat(stats, names=["stat"]).stack(future_stack=True)
    long.index.names = ["stat", "month", "variable"]
    out = long.unstack("stat").reset_index()

    out["total_count"] = out["month"].map(total_count)

    # information value per (variable, month)
    iv_rows = []
    for m in sorted(month.unique()):
        in_m = (month == m).to_numpy()
        y_m = y[in_m]
        valid = y_m.isin([0, 1]).to_numpy()
        y_m = y_m[valid]
        has_both_classes = y_m.nunique() == 2
        sub_m = sub.loc[in_m].loc[valid]
        for c in cols:
            iv = np.nan
            if has_both_classes:
                feature = _bin_feature(sub_m[c], bins, max_cats)
                if feature is not None:
                    tmp = pd.DataFrame({"feature": feature.to_numpy(), "target": y_m.to_numpy()})
                    _, iv = calculate_woe_iv(tmp, "feature", "target")
            iv_rows.append((c, m, iv))
    iv_df = pd.DataFrame(iv_rows, columns=["variable", "month", "iv"])
    out = out.merge(iv_df, on=["variable", "month"], how="left")

    for c in STAT_COLS:
        if c not in out:
            out[c] = float("nan")

    out["month"] = out["month"].map(MONTH_NAMES)
    out["month"] = pd.Categorical(out["month"], ["July", "August", "September"], ordered=True)
    out = out[["variable", "month"] + STAT_COLS]
    return out.sort_values(["variable", "month"]).reset_index(drop=True)

In [ ]:
stability_iv = monthly_stats(df, date_col = 'transaction_time', trans_id_col = "trans_id", year = 2026, target_col = "fraud_label")
stability_iv.to_csv('stability_iv.csv')
stability_iv

### Correlation Matrix

In [ ]:
# Correlation matrix
 
corr = df.select_dtypes("number").corr()               # Pearson
corr = df.select_dtypes("number").corr(method="spearman")  # rank-based, robust to outliers

corr.to_csv('correlation_matrix.csv')

In [ ]:
corr

### Information Value

In [ ]:
# Information value
 
import numpy as np
import pandas as pd


def calculate_woe_iv(df, feature, target):
    # 1. Create a contingency table of counts
    grouped = df.groupby(feature)[target].agg(
        goods=lambda x: (x == 1).sum(), bads=lambda x: (x == 0).sum()
    )

    # 2. Calculate percentages of total goods and bads
    total_goods = grouped["goods"].sum()
    total_bads = grouped["bads"].sum()

    grouped["pct_goods"] = grouped["goods"] / total_goods
    grouped["pct_bads"] = grouped["bads"] / total_bads

    # 3. Handle zero divisions by adding a small adjustment if necessary
    grouped["pct_goods"] = grouped["pct_goods"].replace(0, 0.0001)
    grouped["pct_bads"] = grouped["pct_bads"].replace(0, 0.0001)

    # 4. Calculate WoE and IV
    grouped["WoE"] = np.log(grouped["pct_goods"] / grouped["pct_bads"])
    grouped["IV_bin"] = (grouped["pct_goods"] - grouped["pct_bads"]) * grouped[
        "WoE"
    ]

    total_iv = grouped["IV_bin"].sum()

    return grouped, total_iv


# Example Usage:
data = pd.DataFrame(
    {
        "income_bracket": ["Low", "Low", "Medium", "Medium", "High", "High"]
        * 10,
        "default": [1, 0, 1, 1, 0, 1] * 10,  # 1 = Good, 0 = Bad
    }
)

iv_table, total_iv = calculate_woe_iv(data, "income_bracket", "default")
print(f"Total Information Value: {total_iv:.4f}")
print(iv_table[["WoE", "IV_bin"]])


In [ ]:
def calculate_woe_iv(df, feature, target):
    # 1. Create a contingency table of counts
    grouped = df.groupby(feature)[target].agg(
        goods=lambda x: (x == 1).sum(), bads=lambda x: (x == 0).sum()
    )

    # 2. Calculate percentages of total goods and bads
    total_goods = grouped["goods"].sum()
    total_bads = grouped["bads"].sum()

    grouped["pct_goods"] = grouped["goods"] / total_goods
    grouped["pct_bads"] = grouped["bads"] / total_bads

    # 3. Handle zero divisions by adding a small adjustment if necessary
    grouped["pct_goods"] = grouped["pct_goods"].replace(0, 0.0001)
    grouped["pct_bads"] = grouped["pct_bads"].replace(0, 0.0001)

    # 4. Calculate WoE and IV
    grouped["WoE"] = np.log(grouped["pct_goods"] / grouped["pct_bads"])
    grouped["IV_bin"] = (grouped["pct_goods"] - grouped["pct_bads"]) * grouped[
        "WoE"
    ]

    total_iv = grouped["IV_bin"].sum()

    return grouped, total_iv


def calculate_iv_multiple(df, target, features=None):
    """Apply calculate_woe_iv to several variables and classify each IV."""
    if features is None:
        features = [c for c in df.columns if c != target]

    rows, details = [], {}
    for feature in features:
        table, total_iv = calculate_woe_iv(df, feature, target)
        details[feature] = table
        rows.append({"variable": feature, "n_bins": 5, "IV": total_iv}) #len(table)

    summary = (
        pd.DataFrame(rows)
        .sort_values("IV", ascending=False)
        .reset_index(drop=True)
    )
    summary["strength"] = pd.cut(
        summary["IV"],
        bins=[-np.inf, 0.02, 0.1, 0.3, 0.5, np.inf],
        labels=["useless", "weak", "medium", "strong", "suspicious"],
    )
    return summary, details

summary, details = calculate_iv_multiple(df, "fraud_label")

In [ ]:
summary.to_csv('information_value.csv')

In [ ]:
summary

### PSI

In [ ]:
import numpy as np
import pandas as pd


training_set = df.loc[df['sample_type'] == 'training']
oot_set = df.loc[df['sample_type'] == 'oot']

def psi_all(train, oot, bins=10):
    """PSI of every column of `train` vs `oot` (two DataFrames with the same columns)."""
    result = {}
    for col in train.columns:
        t, o = train[col], oot[col]
 
        # Dates make no sense as categories (every timestamp would be its own bin): skip them
        if pd.api.types.is_datetime64_any_dtype(t):
            continue
 
        # Continuous numeric: bin with training quantiles
        if pd.api.types.is_numeric_dtype(t) and t.nunique() > bins:
            edges = np.unique(np.nanquantile(t, np.linspace(0, 1, bins + 1)))
            edges[0], edges[-1] = -np.inf, np.inf
            t, o = pd.cut(t, edges), pd.cut(o, edges)
 
        # % of rows per bin/category (missing values count as their own bin)
        p = pd.concat(
            [t.value_counts(normalize=True, dropna=False),
             o.value_counts(normalize=True, dropna=False)],
            axis=1, keys=["train", "oot"], sort=False,
        ).fillna(0).clip(lower=1e-4)
 
        result[col] = ((p["oot"] - p["train"]) * np.log(p["oot"] / p["train"])).sum()
 
    out = pd.Series(result, name="PSI").sort_values(ascending=False).to_frame()
 
    # < 0.1 stable | 0.1 - 0.25 moderate shift | > 0.25 significant shift
    out["stability"] = pd.cut(
        out["PSI"],
        bins=[-np.inf, 0.1, 0.25, np.inf],
        labels=["stable", "moderate shift", "significant shift"],
    )
    return out


psi_table = psi_all(training_set, oot_set)
psi_table.to_csv('psi_table.csv')

In [ ]:
psi_table

### Stability

In [ ]:
"""
stats_by_dt.py
--------------
For every numeric variable in a DataFrame, compute per `dt`:
    count, nulls, min, max, mean, median, skew, kurtosis, p1, p10, p25, p50, p75, p90, p99

skew     = population skewness (third standardized moment); 0 = symmetric
kurtosis = EXCESS kurtosis (fourth standardized moment - 3); 0 = normal.
           Both implementations use the same population formulas as Spark's
           skewness()/kurtosis(), so Spark and pandas results match.

Two implementations with the same output format (long format: one row per dt x variable):
    - stats_by_dt_spark(df, ...)   -> for Spark / Databricks DataFrames (scales to big data)
    - stats_by_dt_pandas(df, ...)  -> for pandas DataFrames (small / in-memory data)

Spark usage (always bound the date range and rows to save compute):
    from stats_by_dt import stats_by_dt_spark
    df = (spark.table("catalog.schema.reviews")
          .where((F.col("dt") >= "2026-10-01") & (F.col("dt") < "2026-10-28"))
          .select("dt", "rating", "helpful_votes", "text_len")
          .limit(1_000_000))
    result = stats_by_dt_spark(df)            # pandas DataFrame
    result.to_csv("stats_by_dt.csv", index=False)

Pandas usage:
    result = stats_by_dt_pandas(pdf, dt_col="dt")
"""

from typing import Iterable, List, Optional

import pandas as pd

PERCENTILES = [0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]
PCT_NAMES = ["p1", "p10", "p25", "p50", "p75", "p90", "p99"]
OUTPUT_COLS = ["dt", "variable", "count", "nulls", "min", "max", "mean", "median", "skew", "kurtosis"] + PCT_NAMES


# --------------------------------------------------------------------------- #
# Spark
# --------------------------------------------------------------------------- #
def stats_by_dt_spark(
    df,
    dt_col: str = "dt",
    variables: Optional[Iterable[str]] = None,
    accuracy: int = 10000,
) -> pd.DataFrame:
    """
    Percentiles use percentile_approx (accuracy=10000 -> relative error ~ 1/10000).
    Aggregation runs in a single pass over the data; only the small result is collected.
    """
    import pyspark.sql.functions as F
    from pyspark.sql.types import NumericType

    if variables is None:
        variables = [
            f.name for f in df.schema.fields
            if isinstance(f.dataType, NumericType) and f.name != dt_col
        ]
    variables = list(variables)
    if not variables:
        raise ValueError("No numeric variables found.")

    aggs = []
    for v in variables:
        c = F.col(v).cast("double")
        aggs += [
            F.count(c).alias(f"{v}||count"),
            F.sum(F.col(v).isNull().cast("int")).alias(f"{v}||nulls"),
            F.min(c).alias(f"{v}||min"),
            F.max(c).alias(f"{v}||max"),
            F.avg(c).alias(f"{v}||mean"),
            F.skewness(c).alias(f"{v}||skew"),
            F.kurtosis(c).alias(f"{v}||kurtosis"),
            F.percentile_approx(c, PERCENTILES, accuracy).alias(f"{v}||pcts"),
        ]

    wide = df.groupBy(dt_col).agg(*aggs).orderBy(dt_col).collect()

    rows: List[dict] = []
    for r in wide:
        for v in variables:
            pcts = r[f"{v}||pcts"] or [None] * len(PERCENTILES)
            row = {
                "dt": r[dt_col],
                "variable": v,
                "count": r[f"{v}||count"],
                "nulls": r[f"{v}||nulls"],
                "min": r[f"{v}||min"],
                "max": r[f"{v}||max"],
                "mean": r[f"{v}||mean"],
                "median": pcts[PERCENTILES.index(0.50)],
                "skew": r[f"{v}||skew"],
                "kurtosis": r[f"{v}||kurtosis"],
            }
            row.update(dict(zip(PCT_NAMES, pcts)))
            rows.append(row)

    return pd.DataFrame(rows, columns=OUTPUT_COLS)


# --------------------------------------------------------------------------- #
# pandas
# --------------------------------------------------------------------------- #
def _skew(s: pd.Series) -> float:
    """Population skewness (matches Spark's skewness())."""
    x = s.dropna().astype("float64")
    if len(x) == 0:
        return float("nan")
    d = x - x.mean()
    m2 = (d ** 2).mean()
    return float((d ** 3).mean() / m2 ** 1.5) if m2 > 0 else float("nan")


def _excess_kurtosis(s: pd.Series) -> float:
    """Population excess kurtosis (matches Spark's kurtosis())."""
    x = s.dropna().astype("float64")
    if len(x) == 0:
        return float("nan")
    d = x - x.mean()
    m2 = (d ** 2).mean()
    return float((d ** 4).mean() / m2 ** 2 - 3.0) if m2 > 0 else float("nan")


def stats_by_dt_pandas(
    pdf: pd.DataFrame,
    dt_col: str = "dt",
    variables: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    if variables is None:
        variables = [
            c for c in pdf.select_dtypes(include="number").columns if c != dt_col
        ]
    variables = list(variables)
    if not variables:
        raise ValueError("No numeric variables found.")

    rows: List[dict] = []
    for dt, g in pdf.groupby(dt_col):
        for v in variables:
            s = g[v]
            q = s.quantile(PERCENTILES)
            row = {
                "dt": dt,
                "variable": v,
                "count": int(s.count()),
                "nulls": int(s.isna().sum()),
                "min": s.min(),
                "max": s.max(),
                "mean": s.mean(),
                "median": s.median(),
                "skew": _skew(s),
                "kurtosis": _excess_kurtosis(s),
            }
            row.update({n: q.loc[p] for n, p in zip(PCT_NAMES, PERCENTILES)})
            rows.append(row)

    return pd.DataFrame(rows, columns=OUTPUT_COLS).sort_values(["variable", "dt"]).reset_index(drop=True)


# --------------------------------------------------------------------------- #
# Optional: pivot helper to compare one statistic across dates
# --------------------------------------------------------------------------- #
def pivot_stat(result: pd.DataFrame, stat: str = "p50") -> pd.DataFrame:
    """Rows = variable, columns = dt, values = chosen statistic (e.g. 'mean', 'p99')."""
    return result.pivot(index="variable", columns="dt", values=stat)


if __name__ == "__main__":
    # Quick demo with synthetic data
    import numpy as np

    rng = np.random.default_rng(0)
    n = 3000
    demo = pd.DataFrame(
        {
            "dt": rng.choice(["2026-10-01", "2026-10-02", "2026-10-03"], n),
            "rating": rng.integers(1, 6, n),
            "helpful_votes": rng.poisson(3, n),
            "text_len": rng.lognormal(5, 0.6, n),
        }
    )
    out = stats_by_dt_pandas(demo)
    pd.set_option("display.width", 200)
    print(out.round(2).to_string(index=False))
    print("\nMedian by dt:\n", pivot_stat(out, "median").round(2))